In [1]:
import asyncio
from decimal import Decimal
from datetime import datetime, timedelta, timezone

from typing_extensions import Literal

from typed_bybit import Bybit
from typed_bybit.linear.orderbook import LinearOrderbookUpdate
from typed_bybit.private.execution import ExecutionUpdate
from typed_bybit.spot.orderbook import OrderbookUpdate
from typed_bybit.trade.create_order import (
  CreateLimitOrderRequest,
  CreateMarketOrderRequest,
)
from dotenv import load_dotenv

from tribulnation.sdk.market import (
  Book,
  Collateral,
  PerpCollateral,
  FundingPayment,
  FundingRate,
  NextFunding,
  Order,
  OrderResponse,
  OrderState,
  PerpPosition,
  Position,
  Rules,
  Settings,
  Trade,
)

load_dotenv()

client = await Bybit.new().__aenter__()

MARKETS = {
  'spot': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT'],
  'perp': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT'],
}

Bybit v5 is a *unified* API: spot, linear perps, inverse perps and options are the
same HTTP/WS endpoints, discriminated by a `category` parameter (`'spot'`,
`'linear'`, `'inverse'`, `'option'`) rather than separate namespaces like Binance's
`spot`/`usdm_futures` split. Below, `category='spot'` covers `Market` and
`category='linear'` covers `PerpMarket`, both against the *same* `client.*`.

## `Market` (spot)

In [2]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.market.orderbook(category='spot', symbol=symbol, limit=levels)
  return Book(
    bids=[Book.Entry(p, q) for p, q in raw['b']],
    asks=[Book.Entry(p, q) for p, q in raw['a']],
  )


{symbol: await depth(symbol, levels=5) for symbol in MARKETS['spot']}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('79696.3'), qty=Decimal('0.166824')), Book.Entry(price=Decimal('79696.2'), qty=Decimal('0.021019')), Book.Entry(price=Decimal('79696.1'), qty=Decimal('0.001343')), Book.Entry(price=Decimal('79695.6'), qty=Decimal('0.002')), Book.Entry(price=Decimal('79695.3'), qty=Decimal('0.000919'))], asks=[Book.Entry(price=Decimal('79696.4'), qty=Decimal('0.452447')), Book.Entry(price=Decimal('79696.5'), qty=Decimal('0.012035')), Book.Entry(price=Decimal('79696.8'), qty=Decimal('0.000963')), Book.Entry(price=Decimal('79698.1'), qty=Decimal('0.001605')), Book.Entry(price=Decimal('79698.9'), qty=Decimal('0.024911'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2452.74'), qty=Decimal('7.00655')), Book.Entry(price=Decimal('2452.73'), qty=Decimal('0.01936')), Book.Entry(price=Decimal('2452.68'), qty=Decimal('0.03533')), Book.Entry(price=Decimal('2452.63'), qty=Decimal('0.03872')), Book.Entry(price=Decimal('2452.59'), qty=Decimal('0.81544'))], asks=[B

In [3]:
def depth_stream(symbol: str, *, depth: Literal[1, 50, 200] = 50):
  # Bybit's WS orderbook channel pushes one full snapshot on subscribe, then
  # incremental deltas: only the changed price levels, with qty "0" meaning the
  # level was removed. Merge every push into a running `Book` via `Book.update`
  # (adds/replaces changed levels, drops zero-qty ones) instead of treating each
  # push as if it were a complete book -- a delta only has a handful of levels,
  # not the full depth. `.copy()` each yielded book so earlier snapshots handed to
  # callers aren't mutated by later pushes into the same running `book`.
  book = Book()

  def to_book(update: OrderbookUpdate) -> Book:
    delta = Book(
      bids=[Book.Entry(p, q) for p, q in update['b']],
      asks=[Book.Entry(p, q) for p, q in update['a']],
    )
    book.update(delta)
    return book.copy()

  return client.spot.orderbook(depth, symbol=symbol).map(to_book)


books: list[Book] = []
async with depth_stream('BTCUSDT') as stream:
  async for book in stream:
    books.append(book)
    if len(books) >= 3:
      break
books

[Book(bids=[Book.Entry(price=Decimal('79696.3'), qty=Decimal('0.221024')), Book.Entry(price=Decimal('79696.2'), qty=Decimal('0.021019')), Book.Entry(price=Decimal('79696.1'), qty=Decimal('0.001343')), Book.Entry(price=Decimal('79695.6'), qty=Decimal('0.004')), Book.Entry(price=Decimal('79695.3'), qty=Decimal('0.000919')), Book.Entry(price=Decimal('79695.2'), qty=Decimal('0.043779')), Book.Entry(price=Decimal('79695.1'), qty=Decimal('0.024506')), Book.Entry(price=Decimal('79694'), qty=Decimal('0.001531')), Book.Entry(price=Decimal('79693.6'), qty=Decimal('0.023681')), Book.Entry(price=Decimal('79692.7'), qty=Decimal('0.001838')), Book.Entry(price=Decimal('79692.5'), qty=Decimal('0.01')), Book.Entry(price=Decimal('79692.4'), qty=Decimal('0.051606')), Book.Entry(price=Decimal('79691.7'), qty=Decimal('0.004')), Book.Entry(price=Decimal('79691.4'), qty=Decimal('0.001838')), Book.Entry(price=Decimal('79691.3'), qty=Decimal('0.010971')), Book.Entry(price=Decimal('79690'), qty=Decimal('0.00013

In [4]:
async def rules(symbol: str, *, refetch: bool = False) -> Rules:
  info = await client.market.instruments(category='spot', symbol=symbol)
  # `instruments()` returns a 3-way union (`Spot|Contract|OptionInstrumentsInfo`)
  # regardless of the `category` argument -- narrow it by hand via the shared
  # `category` discriminant, same idiom as every other `category='spot'`/`'linear'`
  # call below.
  assert info['category'] == 'spot'
  sym = info['list'][0]
  lot = sym['lotSizeFilter']
  price_filter = sym['priceFilter']
  fee = await client.account.fee_rate(category='spot', symbol=symbol)
  rate = fee['list'][0] if fee['list'] else None
  return Rules(
    base=sym['baseCoin'],
    quote=sym['quoteCoin'],
    fee_asset=sym['quoteCoin'],
    tick_size=price_filter['tickSize'],
    step_size=lot['basePrecision'],
    fixed_min_qty=lot['minOrderQty'],
    min_value=lot['minOrderAmt'],
    max_qty=lot['maxOrderQty'],
    maker_fee=rate['makerFeeRate'] if rate else Decimal(0),
    taker_fee=rate['takerFeeRate'] if rate else Decimal(0),
    api=sym['status'] == 'Trading',
    details=sym,
  )


{symbol: await rules(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.1'), step_size=Decimal('0.000001'), fixed_min_qty=Decimal('0.000001'), min_value=Decimal('5'), max_qty=Decimal('230'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.001'), taker_fee=Decimal('0.001'), api=True, details={'symbolId': 9, 'symbol': 'BTCUSDT', 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'innovation': '0', 'status': 'Trading', 'marginTrading': 'utaOnly', 'stTag': '0', 'lotSizeFilter': {'basePrecision': Decimal('0.000001'), 'quotePrecision': '0.0000001', 'minOrderQty': Decimal('0.000001'), 'maxOrderQty': Decimal('230'), 'minOrderAmt': Decimal('5'), 'maxOrderAmt': '8000000', 'maxLimitOrderQty': '230', 'maxMarketOrderQty': '120', 'postOnlyMaxLimitOrderSize': '1150'}, 'priceFilter': {'tickSize': Decimal('0.1')}, 'riskParameters': {'priceLimitRatioX': '0.005', 'priceLimitRatioY': '0.01'}, 'symbolType': '', 'xstockMultiplier': '1'}),
 'ETHUSDT': 

In [5]:
async def open_orders(symbol: str) -> list[OrderState]:
  raw = await client.trade.open_orders(category='spot', symbol=symbol)
  out: list[OrderState] = []
  for o in raw['list']:
    qty = Decimal(o['qty'])
    filled = Decimal(o['cumExecQty'])
    sign = 1 if o['side'] == 'Buy' else -1
    out.append(
      OrderState(
        id=o['orderId'],
        price=Decimal(o['price']),
        qty=sign * qty,
        filled_qty=sign * filled,
        active=o['orderStatus'] in ('New', 'PartiallyFilled', 'Untriggered'),
        details=o,
      )
    )
  return out


{symbol: await open_orders(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [6]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.trade.trade_history(
    category='spot',
    symbol=symbol,
    start_time=start,
    end_time=end,
    exec_type='Trade',
  )
  out: list[Trade] = []
  for t in raw['list']:
    qty = Decimal(t['execQty'])
    out.append(
      Trade(
        id=t['execId'],
        price=Decimal(t['execPrice']),
        qty=qty if t['side'] == 'Buy' else -qty,
        time=t['execTime'],
        maker=t['isMaker'],
        # `execFee` is a required, already-parsed `Decimal`. A truthiness guard here
        # would be wrong, not merely redundant: 40 of the 55 spot fills this account
        # has ever made carry `execFee` exactly `0`, and reporting those as `fee=None`
        # ("unknown") rather than a zero fee loses real information.
        fee=Trade.Fee(amount=t['execFee'], asset=t['feeCurrency']),
        details=t,
      )
    )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await trades_history(symbol, start, end) for symbol in MARKETS['spot']}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [3]:
def trades_stream(symbol: str, *, category: str = 'spot'):
  # Bybit's private `execution` channel carries every category on one connection;
  # filter client-side by category/symbol/execType, like every other unified-API stream.
  def parse(execs: list[ExecutionUpdate]) -> list[Trade]:
    out: list[Trade] = []
    for e in execs:
      if e['category'] != category or e['symbol'] != symbol or e['execType'] != 'Trade':
        continue
      qty = Decimal(e['execQty'])
      out.append(
        Trade(
          id=e['execId'],
          price=Decimal(e['execPrice']),
          qty=qty if e['side'] == 'Buy' else -qty,
          time=e['execTime'],
          maker=e['isMaker'],
          # `execFee` is required here too, and arrives as an unparsed `str` on the WS
          # channel rather than the REST endpoint's `Decimal` -- see `trades_history`
          # above for why a zero fee must not collapse to `None`.
          fee=Trade.Fee(amount=Decimal(e['execFee']), asset=e['feeCurrency']),
          details=e,
        )
      )
    return out

  return client.private.execution().map(parse).filter(lambda trades: len(trades) > 0)


async with trades_stream('BTCUSDT') as stream:
  it = aiter(stream)
  try:
    result = await asyncio.wait_for(anext(it), timeout=5.0)
  except asyncio.TimeoutError:
    result = (
      'no new trades observed in 5s (expected -- no live trading on this account)'
    )
result

'no new trades observed in 5s (expected -- no live trading on this account)'

In [7]:
async def position(symbol: str) -> Position:
  info = await client.market.instruments(category='spot', symbol=symbol)
  assert info['category'] == 'spot'  # narrow the 3-way union -- see `rules()` above
  base = info['list'][0]['baseCoin']
  wallet = await client.account.wallet_balance(account_type='UNIFIED', coin=base)
  coins = wallet['list'][0]['coin'] if wallet['list'] else []
  bal = next((c for c in coins if c['coin'] == base), None)
  # Every numeric field is `''` for a coin the account has never held -- guard that
  # before reading it as a `Decimal`.
  size = bal['walletBalance'] if bal and bal['walletBalance'] else Decimal(0)
  return Position(size=size)


{symbol: await position(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Position(size=Decimal('0')),
 'ETHUSDT': Position(size=Decimal('0')),
 'SOLUSDT': Position(size=Decimal('0'))}

In [8]:
async def collateral(symbol: str) -> Collateral:
  info = await client.market.instruments(category='spot', symbol=symbol)
  assert info['category'] == 'spot'  # narrow the 3-way union -- see `rules()` above
  quote = info['list'][0]['quoteCoin']
  wallet = await client.account.wallet_balance(account_type='UNIFIED', coin=quote)
  coins = wallet['list'][0]['coin'] if wallet['list'] else []
  bal = next((c for c in coins if c['coin'] == quote), None)
  equity = bal['walletBalance'] if bal and bal['walletBalance'] else Decimal(0)
  # `locked` is `NotRequired`; `'locked' in bal` (not `bal.get('locked')`) is what
  # lets pyright narrow it away before the direct-index read below.
  locked = bal['locked'] if bal and 'locked' in bal and bal['locked'] else Decimal(0)
  return Collateral(equity=equity, free_collateral=equity - locked)


{symbol: await collateral(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Collateral(equity=Decimal('0.0005584'), free_collateral=Decimal('0.0005584')),
 'ETHUSDT': Collateral(equity=Decimal('0.0005584'), free_collateral=Decimal('0.0005584')),
 'SOLUSDT': Collateral(equity=Decimal('0.0005584'), free_collateral=Decimal('0.0005584'))}

In [9]:
async def available_notional(symbol: str) -> Decimal:
  c = await collateral(symbol)
  return c.free_collateral


{symbol: await available_notional(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Decimal('0.0005584'),
 'ETHUSDT': Decimal('0.0005584'),
 'SOLUSDT': Decimal('0.0005584')}

In [10]:
async def place_order(
  symbol: str, order: Order, *, settings: Settings = {}
) -> OrderResponse:
  order_qty = Decimal(order['qty'])
  side: Literal['Buy', 'Sell'] = 'Buy' if order_qty > 0 else 'Sell'
  qty = str(abs(order_qty))
  time_in_force: Literal['GTC', 'PostOnly'] = (
    'PostOnly' if order['type'] == 'POST_ONLY' else 'GTC'
  )
  body: CreateMarketOrderRequest | CreateLimitOrderRequest
  if order['type'] == 'MARKET':
    body = {
      'category': 'spot',
      'symbol': symbol,
      'side': side,
      'orderType': 'Market',
      'qty': qty,
      'timeInForce': time_in_force,
    }
  else:
    body = {
      'category': 'spot',
      'symbol': symbol,
      'side': side,
      'orderType': 'Limit',
      'qty': qty,
      'timeInForce': time_in_force,
      'price': str(Decimal(order['price'])),
    }
  raw = await client.trade.create_order(body)
  return OrderResponse(id=raw['orderId'], details=raw)


# Not executed here -- would place a real order on the account.
await place_order(
  'BTCUSDT', {'qty': Decimal('0.0001'), 'price': Decimal('20000'), 'type': 'LIMIT'}
)

ApiError: ApiError(170140, Order value exceeded lower limit., {'retCode': 170140, 'retMsg': 'Order value exceeded lower limit.', 'result': {}, 'retExtInfo': {}, 'time': 1788549064798})

In [11]:
async def cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.trade.cancel_order(category='spot', symbol=symbol, order_id=id)


# Not executed here -- would cancel a real order on the account.
await cancel_order('BTCUSDT', '123456')

ApiError: ApiError(170213, Order does not exist., {'retCode': 170213, 'retMsg': 'Order does not exist.', 'result': {}, 'retExtInfo': {}, 'time': 1788549068682})

## `PerpMarket` (linear)

In [12]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.market.orderbook(category='linear', symbol=symbol, limit=levels)
  return Book(
    bids=[Book.Entry(p, q) for p, q in raw['b']],
    asks=[Book.Entry(p, q) for p, q in raw['a']],
  )


{symbol: await depth(symbol, levels=5) for symbol in MARKETS['perp']}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('79637.60'), qty=Decimal('8.095')), Book.Entry(price=Decimal('79637.50'), qty=Decimal('2.638')), Book.Entry(price=Decimal('79637.40'), qty=Decimal('0.490')), Book.Entry(price=Decimal('79637.30'), qty=Decimal('0.001')), Book.Entry(price=Decimal('79637.10'), qty=Decimal('0.002'))], asks=[Book.Entry(price=Decimal('79637.70'), qty=Decimal('2.037')), Book.Entry(price=Decimal('79637.80'), qty=Decimal('0.001')), Book.Entry(price=Decimal('79638.20'), qty=Decimal('0.001')), Book.Entry(price=Decimal('79638.60'), qty=Decimal('0.001')), Book.Entry(price=Decimal('79639.00'), qty=Decimal('0.005'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2450.76'), qty=Decimal('31.10')), Book.Entry(price=Decimal('2450.75'), qty=Decimal('0.01')), Book.Entry(price=Decimal('2450.74'), qty=Decimal('0.02')), Book.Entry(price=Decimal('2450.72'), qty=Decimal('0.02')), Book.Entry(price=Decimal('2450.71'), qty=Decimal('0.02'))], asks=[Book.Entry(price=Decimal('2450.7

In [12]:
def depth_stream(symbol: str, *, depth: Literal[1, 50, 200, 1000] = 50):
  # Same snapshot-then-deltas merge as the spot `depth_stream` above -- see its
  # comment for why treating every push as a full book is wrong.
  book = Book()

  def to_book(update: LinearOrderbookUpdate) -> Book:
    delta = Book(
      bids=[Book.Entry(p, q) for p, q in update['b']],
      asks=[Book.Entry(p, q) for p, q in update['a']],
    )
    book.update(delta)
    return book.copy()

  return client.linear.orderbook(depth, symbol=symbol).map(to_book)


books: list[Book] = []
async with depth_stream('BTCUSDT') as stream:
  async for book in stream:
    books.append(book)
    if len(books) >= 3:
      break
books

[Book(bids=[Book.Entry(price=Decimal('80794.00'), qty=Decimal('1.053')), Book.Entry(price=Decimal('80793.90'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80793.20'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80793.10'), qty=Decimal('0.002')), Book.Entry(price=Decimal('80793.00'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80792.70'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80792.60'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80792.00'), qty=Decimal('0.003')), Book.Entry(price=Decimal('80791.30'), qty=Decimal('0.020')), Book.Entry(price=Decimal('80791.20'), qty=Decimal('0.008')), Book.Entry(price=Decimal('80790.90'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80790.50'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80790.40'), qty=Decimal('0.002')), Book.Entry(price=Decimal('80790.30'), qty=Decimal('0.006')), Book.Entry(price=Decimal('80790.10'), qty=Decimal('0.002')), Book.Entry(price=Decimal('80790.00'), qty=Decimal('0.044')), Book.Entry(p

In [13]:
async def rules(symbol: str, *, refetch: bool = False) -> Rules:
  info = await client.market.instruments(category='linear', symbol=symbol)
  assert info['category'] == 'linear'  # narrow the 3-way union -- see spot `rules()`
  sym = info['list'][0]
  lot = sym['lotSizeFilter']
  price_filter = sym['priceFilter']
  fee = await client.account.fee_rate(category='linear', symbol=symbol)
  rate = fee['list'][0] if fee['list'] else None
  return Rules(
    base=sym['baseCoin'],
    quote=sym['quoteCoin'],
    fee_asset=sym['settleCoin'],
    tick_size=price_filter['tickSize'],
    step_size=lot['qtyStep'],
    fixed_min_qty=lot['minOrderQty'],
    min_value=lot['minNotionalValue'],
    max_qty=lot['maxOrderQty'],
    fixed_min_price=price_filter['minPrice'],
    fixed_max_price=price_filter['maxPrice'],
    maker_fee=rate['makerFeeRate'] if rate else Decimal(0),
    taker_fee=rate['takerFeeRate'] if rate else Decimal(0),
    api=sym['status'] == 'Trading',
    details=sym,
  )


{symbol: await rules(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.10'), step_size=Decimal('0.001'), fixed_min_qty=Decimal('0.001'), min_value=Decimal('5'), max_qty=Decimal('1500.000'), fixed_min_price=Decimal('0.10'), rel_min_price=None, rel_max_price=None, fixed_max_price=Decimal('1999999.80'), maker_fee=Decimal('0.0002'), taker_fee=Decimal('0.00055'), api=True, details={'symbol': 'BTCUSDT', 'symbolId': 5, 'contractType': 'LinearPerpetual', 'status': 'Trading', 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'launchTime': datetime.datetime(2020, 3, 15, 0, 0), 'deliveryTime': datetime.datetime(1970, 1, 1, 0, 0), 'deliveryFeeRate': '', 'priceScale': '2', 'leverageFilter': {'minLeverage': Decimal('1'), 'maxLeverage': Decimal('150.00'), 'leverageStep': Decimal('0.01')}, 'priceFilter': {'minPrice': Decimal('0.10'), 'maxPrice': Decimal('1999999.80'), 'tickSize': Decimal('0.10')}, 'lotSizeFilter': {'maxOrderQty': Decimal('1500.000'), 'minOrderQty': Decimal('0.001'), 'qtyStep': Decim

In [14]:
async def open_orders(symbol: str) -> list[OrderState]:
  raw = await client.trade.open_orders(category='linear', symbol=symbol)
  out: list[OrderState] = []
  for o in raw['list']:
    qty = Decimal(o['qty'])
    filled = Decimal(o['cumExecQty'])
    sign = 1 if o['side'] == 'Buy' else -1
    out.append(
      OrderState(
        id=o['orderId'],
        price=Decimal(o['price']),
        qty=sign * qty,
        filled_qty=sign * filled,
        active=o['orderStatus'] in ('New', 'PartiallyFilled', 'Untriggered'),
        details=o,
      )
    )
  return out


{symbol: await open_orders(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [2]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.trade.trade_history(
    category='linear',
    symbol=symbol,
    start_time=start,
    end_time=end,
    exec_type='Trade',
  )
  out: list[Trade] = []
  for t in raw['list']:
    qty = Decimal(t['execQty'])
    out.append(
      Trade(
        id=t['execId'],
        price=Decimal(t['execPrice']),
        qty=qty if t['side'] == 'Buy' else -qty,
        time=t['execTime'],
        maker=t['isMaker'],
        # Same `execFee` handling as the spot `trades_history` above, off the same
        # endpoint and the same required `Decimal` field -- though the reasoning there is
        # drawn from real spot fills, since this account has never taken a linear one.
        fee=Trade.Fee(amount=t['execFee'], asset=t['feeCurrency']),
        details=t,
      )
    )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await trades_history(symbol, start, end) for symbol in MARKETS['perp']}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [5]:
async with trades_stream('BTCUSDT', category='linear') as stream:
  it = aiter(stream)
  try:
    result = await asyncio.wait_for(anext(it), timeout=5.0)
  except asyncio.TimeoutError:
    result = (
      'no new trades observed in 5s (expected -- no live trading on this account)'
    )
result

'no new trades observed in 5s (expected -- no live trading on this account)'

In [17]:
async def index(symbol: str, *, settings: Settings = {}) -> Decimal:
  raw = await client.market.tickers(category='linear', symbol=symbol)
  # `tickers()` returns a 3-way union (`Spot|Contract|OptionTickers`) regardless of
  # `category` -- narrow it by hand via the shared `category` discriminant.
  assert raw['category'] == 'linear'
  return raw['list'][0]['indexPrice']


{symbol: await index(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': Decimal('80835.18'),
 'ETHUSDT': Decimal('2516.19'),
 'SOLUSDT': Decimal('103.605')}

In [18]:
async def next_funding(symbol: str) -> NextFunding:
  ticker_result = await client.market.tickers(category='linear', symbol=symbol)
  assert ticker_result['category'] == 'linear'  # see `index()` above
  ticker = ticker_result['list'][0]
  instruments_result = await client.market.instruments(category='linear', symbol=symbol)
  assert instruments_result['category'] == 'linear'  # see spot `rules()` above
  info = instruments_result['list'][0]
  return NextFunding(
    rate=ticker['fundingRate'],
    time=ticker['nextFundingTime'],
    interval=timedelta(minutes=info['fundingInterval']),
  )


{symbol: await next_funding(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': NextFunding(rate=Decimal('0.00003145'), time=datetime.datetime(2026, 9, 4, 16, 0), premium=None, interval=datetime.timedelta(seconds=28800)),
 'ETHUSDT': NextFunding(rate=Decimal('0.00009267'), time=datetime.datetime(2026, 9, 4, 16, 0), premium=None, interval=datetime.timedelta(seconds=28800)),
 'SOLUSDT': NextFunding(rate=Decimal('-0.00007017'), time=datetime.datetime(2026, 9, 4, 16, 0), premium=None, interval=datetime.timedelta(seconds=28800))}

In [19]:
async def funding_rates(
  symbol: str,
  start: datetime | None = None,
  end: datetime | None = None,
) -> list[FundingRate]:
  raw = await client.market.funding_history(
    category='linear',
    symbol=symbol,
    start_time=start,
    end_time=end,
  )
  return [
    FundingRate(rate=r['fundingRate'], time=r['fundingRateTimestamp'])
    for r in raw['list']
  ]


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
{symbol: await funding_rates(symbol, start, end) for symbol in MARKETS['perp']}

{'BTCUSDT': [FundingRate(rate=Decimal('0.00006921'), time=datetime.datetime(2026, 9, 4, 8, 0), premium=None),
  FundingRate(rate=Decimal('0.00007352'), time=datetime.datetime(2026, 9, 4, 0, 0), premium=None),
  FundingRate(rate=Decimal('0.00008434'), time=datetime.datetime(2026, 9, 3, 16, 0), premium=None),
  FundingRate(rate=Decimal('0.00009388'), time=datetime.datetime(2026, 9, 3, 8, 0), premium=None),
  FundingRate(rate=Decimal('0.00006847'), time=datetime.datetime(2026, 9, 3, 0, 0), premium=None),
  FundingRate(rate=Decimal('0.00005918'), time=datetime.datetime(2026, 9, 2, 16, 0), premium=None),
  FundingRate(rate=Decimal('0.00008602'), time=datetime.datetime(2026, 9, 2, 8, 0), premium=None),
  FundingRate(rate=Decimal('0.00002041'), time=datetime.datetime(2026, 9, 2, 0, 0), premium=None),
  FundingRate(rate=Decimal('0.00002407'), time=datetime.datetime(2026, 9, 1, 16, 0), premium=None),
  FundingRate(rate=Decimal('0.0001'), time=datetime.datetime(2026, 9, 1, 8, 0), premium=None),


In [6]:
async def funding_payments(
  symbol: str, start: datetime, end: datetime
) -> list[FundingPayment]:
  # `account.transaction_log` has no `symbol` filter -- filter client-side.
  raw = await client.account.transaction_log(
    category='linear',
    type='SETTLEMENT',
    start_time=start,
    end_time=end,
  )
  # `funding` is declared `NotRequired[str]`; `'funding' in e` (not `e.get('funding')`)
  # is what lets pyright narrow it away before the direct-index read below. Whether it
  # is ever actually absent or empty on a `type='SETTLEMENT'` row could not be checked:
  # this account has never held a perp position, so `transaction_log` returns no
  # SETTLEMENT row anywhere in the two years Bybit will serve.
  return [
    FundingPayment(amount=-Decimal(e['funding']), time=e['transactionTime'])
    for e in raw['list']
    if e['symbol'] == symbol and 'funding' in e and e['funding']
  ]


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
{symbol: await funding_payments(symbol, start, end) for symbol in MARKETS['perp']}

Task was destroyed but it is pending!
task: <Task pending name='Task-36' coro=<Queue.get() done, defined at /home/ubuntu/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/asyncio/queues.py:149> wait_for=<Future cancelled>>


Task was destroyed but it is pending!
task: <Task pending name='Task-51' coro=<Queue.get() done, defined at /home/ubuntu/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/asyncio/queues.py:149> wait_for=<Future cancelled>>


{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [4]:
async def perp_position(symbol: str) -> PerpPosition:
  raw = await client.position.list(category='linear', symbol=symbol)
  row = raw['list'][0] if raw['list'] else None
  # A symbol with no position still comes back as a row, with `side` and every
  # numeric field an empty string.
  if row is None or not row['side']:
    return PerpPosition()
  size = Decimal(row['size'])
  return PerpPosition(
    size=size if row['side'] == 'Buy' else -size,
    entry_price=Decimal(row['avgPrice']) if row['avgPrice'] else Decimal(0),
  )


{symbol: await perp_position(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'ETHUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'SOLUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0'))}

In [5]:
async def perp_collateral(symbol: str) -> PerpCollateral:
  wallet = (await client.account.wallet_balance(account_type='UNIFIED'))['list'][0]
  info = await client.account.info()
  positions = (await client.position.list(category='linear', settle_coin='USDT'))[
    'list'
  ]
  equity = Decimal(wallet['totalEquity'])
  free_collateral = Decimal(wallet['totalAvailableBalance'])
  initial_margin = Decimal(wallet['totalInitialMargin'])
  maintenance_margin = Decimal(wallet['totalMaintenanceMargin'])
  total_notional = sum(
    (abs(Decimal(p['positionValue'])) for p in positions if p['positionValue']),
    Decimal(0),
  )
  leverage = total_notional / equity if equity > 0 else Decimal(0)
  return PerpCollateral(
    equity=equity,
    free_collateral=free_collateral,
    initial_margin=initial_margin,
    maintenance_margin=maintenance_margin,
    leverage=leverage,
    margin_mode='isolated' if info['marginMode'] == 'ISOLATED_MARGIN' else 'cross',
  )


{symbol: await perp_collateral(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': PerpCollateral(equity=Decimal('0.00055835'), free_collateral=Decimal('0.00055835'), initial_margin=Decimal('0'), maintenance_margin=Decimal('0'), leverage=Decimal('0E+8'), margin_mode='cross'),
 'ETHUSDT': PerpCollateral(equity=Decimal('0.00055835'), free_collateral=Decimal('0.00055835'), initial_margin=Decimal('0'), maintenance_margin=Decimal('0'), leverage=Decimal('0E+8'), margin_mode='cross'),
 'SOLUSDT': PerpCollateral(equity=Decimal('0.00055835'), free_collateral=Decimal('0.00055835'), initial_margin=Decimal('0'), maintenance_margin=Decimal('0'), leverage=Decimal('0E+8'), margin_mode='cross')}

In [23]:
async def available_notional(symbol: str) -> Decimal:
  c = await perp_collateral(symbol)
  instruments_result = await client.market.instruments(category='linear', symbol=symbol)
  assert instruments_result['category'] == 'linear'  # see spot `rules()` above
  info = instruments_result['list'][0]
  max_leverage = info['leverageFilter']['maxLeverage']
  return c.free_collateral * max_leverage


{symbol: await available_notional(symbol) for symbol in MARKETS['perp']}

{'BTCUSDT': Decimal('0.0837525000'),
 'ETHUSDT': Decimal('0.0837525000'),
 'SOLUSDT': Decimal('0.0558350000')}

In [ ]:
async def place_order(
  symbol: str, order: Order, *, settings: Settings = {}
) -> OrderResponse:
  order_qty = Decimal(order['qty'])
  side: Literal['Buy', 'Sell'] = 'Buy' if order_qty > 0 else 'Sell'
  qty = str(abs(order_qty))
  time_in_force: Literal['GTC', 'PostOnly'] = (
    'PostOnly' if order['type'] == 'POST_ONLY' else 'GTC'
  )
  body: CreateMarketOrderRequest | CreateLimitOrderRequest
  if order['type'] == 'MARKET':
    body = {
      'category': 'linear',
      'symbol': symbol,
      'side': side,
      'orderType': 'Market',
      'qty': qty,
      'timeInForce': time_in_force,
    }
  else:
    body = {
      'category': 'linear',
      'symbol': symbol,
      'side': side,
      'orderType': 'Limit',
      'qty': qty,
      'timeInForce': time_in_force,
      'price': str(Decimal(order['price'])),
    }
  raw = await client.trade.create_order(body)
  return OrderResponse(id=raw['orderId'], details=raw)


# Not executed here -- would place a real order on the account.
await place_order(
  'BTCUSDT', {'qty': Decimal('0.001'), 'price': Decimal('20000'), 'type': 'LIMIT'}
)

In [ ]:
async def cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.trade.cancel_order(category='linear', symbol=symbol, order_id=id)


# Not executed here -- would cancel a real order on the account.
await cancel_order('BTCUSDT', '123456')

### Coverage assessment

**Fully supported**, once the unified `category` parameter is threaded through
every call. `Market` and `PerpMarket` reach the *same* `client.market`,
`client.trade`, `client.position` and `client.account` endpoints as
spot -- only `category` (and, for perps, `settleCoin`/`leverageFilter`/funding
fields) changes; there is no separate "futures client" to instantiate, unlike
Binance's `spot`/`usdm_futures` split.

Bybit writes `''` instead of a number wherever a row means "nothing here", and the
client's schemas admit the sentinel, so both shapes of it validate as they stand.
A flat (zero-size) linear position still comes back as a full row from
`position.list` -- `BTCUSDT` here returns `side=''`, `size='0'` and 22 empty-string
fields in all, among them `avgPrice`, `positionValue`, `markPrice`, `positionStatus`,
`createdTime`, `updatedTime` and `tpslMode`; those are declared `Literal[''] | str`
(or `Literal[''] | TimestampMillis` for the timestamps), so the row parses and the
empty string reaches the caller. `account.wallet_balance` does the same for a coin
the account has never held (`SOL` here): `walletBalance`, `equity`, `locked`,
`unrealisedPnl` and the rest come back `''` against a declared
`Literal[''] | Decimal` -- though not quite *every* numeric field, since
`spotHedgingQty` and `spotBorrow` still return `0`. What's left for a caller is
treating the empty string as zero, which `position()`, `collateral()` and
`perp_position()` above each do before converting to a `Decimal`.

`index()` and `next_funding()` are both served for free by `market.tickers`
(`indexPrice`, `fundingRate`, `nextFundingTime`) -- no dedicated index-price
endpoint call was needed. `funding_payments` has no per-symbol filter on
`account.transaction_log`, so it's filtered client-side; the same 7-day window cap
applies to both `funding_rates`/`funding_payments` (`transaction_log`) and general
trade history.

One caveat that costs nothing here but would elsewhere: `typed_bybit` parses every
`TimestampMillis` field into a *naive* `datetime`, and the market types above are
plain dataclasses that accept it without complaint -- so `Trade.time`,
`FundingRate.time` and `NextFunding.time` all carry naive values, shifted into local
time on a host that isn't UTC. `reporting.ipynb` has to stamp the timezone back on,
because its observations are pydantic models that reject a naive datetime.

Two branches above are written but unexercised, since this account holds no perp
position and never has: the `avgPrice` fallback in `perp_position()` and the
`positionValue` filter in `perp_collateral()`. `position.list(settle_coin='USDT')`
returns only positions with a non-zero size, so it currently returns nothing at all.
